# 04 — Query 4: Spatial Analysis & Coastal Hotspot Clustering

- **Owner:** Louis Le Forestier (`@louislefo`)
- **Course Rubric:** Query 4 `[2 pts]` (reusing Lab 8 `GeoPandas`, `shapely`, `sjoin`, `KMeans`, and `OPTICS`)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, clean_shark_data
from src.indicators import geocode_coastal_areas, query_spatial_clustering

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
GEO_CACHE = '../data/locations_geocoded.csv'
SHP_PATH = '../data/countries_shp/ne_110m_admin_0_countries.shp'

## Query 4 — GeoPandas Spatial Join, K-Means Basins & OPTICS Density Hotspots `[2 pts]`

### Indicator Explanation
- **Calculation:**
  1. Enriched `(Country_clean, Area_clean)` pairs with coastal `(latitude, longitude)` coordinates via `geocode_coastal_areas()` (area-level coordinates for major coastal states/provinces + country coastal centroids fallback, cached locally in `projet/data/locations_geocoded.csv`).
  2. Constructed a `GeoDataFrame` (`Point(longitude, latitude)`, `CRS = EPSG:4326`) and performed a spatial join (`gpd.sjoin`) against Natural Earth `ne_110m_admin_0_countries.shp` (Lab 8 pattern).
  3. Ran **K-Means** ($k = 6$, `random_state = 42`) with **dynamic centroid-based cluster labeling** (no hardcoded index assumptions) and compared it against **OPTICS** density-based clustering (`min_samples = 25`, `xi = 0.05`) to isolate dense coastal hubs from scattered outlier coasts (`cluster = -1`).
- **What it represents:** Maps global coastal epicenters of shark-human encounters and compares regional severity profiles.
- **Interpretation:** Over **95%** of cleaned attacks are geocoded into 6 distinct oceanic basins. The **US Atlantic / Florida / Caribbean** basin concentrates the highest volume of incidents with a low fatality rate (~5%), whereas **Australia / South-West Pacific** (~18%) and **Southern Africa / Indian Ocean** (~17%) display dramatically higher lethality due to deeper coastal waters and larger predatory species (White, Tiger, and Bull sharks).

In [2]:
# Build/load coastal coordinates cache and run GeoPandas + K-Means + OPTICS clustering
geo_coords = geocode_coastal_areas(clean_sharks, cache_path=GEO_CACHE, force_refresh=True)
q4 = query_spatial_clustering(clean_sharks, geo_coords, shp_path=SHP_PATH, n_clusters=6)

print(f"Geocoding coverage: {q4['coverage_pct']}% ({len(q4['geo_df'])} / {len(clean_sharks)} incidents)")
print(f"GeoPandas sjoin land-polygon matches: {q4['sjoin_matched']} points")
print('\n--- K-Means Spatial Cluster Profiles (Dynamic Centroid Labels) ---')
display(q4['cluster_summary'])
q4['fig_map'].show()

Geocoding coverage: 95.59% (3855 / 4033 incidents)
GeoPandas sjoin land-polygon matches: 2928 points

--- K-Means Spatial Cluster Profiles (Dynamic Centroid Labels) ---


,Cluster_Name,Total_Incidents,Fatal_Count,Mean_Lat,Mean_Lon,Dominant_Activity,Fatality_Rate_Pct
0,"US Atlantic, Florida & Caribbean (United State...",1591,105,25.31,-77.96,Surfing,6.60
1,Australia & South-West Pacific (Australia / Ne...,797,129,-29.46,154.31,Surfing,16.19
2,Southern Africa & Indian Ocean (South Africa /...,544,94,-30.14,29.60,Surfing,17.28
3,Pacific & US West Coast / Hawaii (United State...,527,38,25.47,-138.05,Surfing,7.21
4,Australia & South-West Pacific (Australia / Ph...,270,65,-8.75,117.80,Diving/Snorkeling,24.07
5,"Europe, Mediterranean & Red Sea (Italy / Egypt...",126,18,38.69,15.16,Diving/Snorkeling,14.29


In [3]:
# Inspect OPTICS density-based spatial clustering comparison (Lab 8)
print('--- OPTICS Density Cluster Summary ---')
display(q4['optics_summary'])
q4['fig_optics_map'].show()

--- OPTICS Density Cluster Summary ---


,OPTICS_Label,Total_Incidents,Top_Country
0,Noise / Isolated Coast (-1),1434,Australia
1,Dense Hotspot #2,919,United States
2,Dense Hotspot #4,274,South Africa
3,Dense Hotspot #7,243,Australia
4,Dense Hotspot #0,233,United States
5,Dense Hotspot #3,222,United States
6,Dense Hotspot #1,207,United States
7,Dense Hotspot #6,165,Australia
8,Dense Hotspot #5,158,South Africa
